In [45]:
import os
import random
import shutil
import numpy as np
import tensorflow as tf

from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.13.0


In [46]:
# notebook is in MRI/notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

RAW_DATASET = os.path.join(PROJECT_ROOT, "dataset")        # PD / NonPD
SPLIT_DATASET = os.path.join(PROJECT_ROOT, "dataset_split")
MODEL_DIR = os.path.join(PROJECT_ROOT, "model")

IMG_SIZE = (192, 192)
BATCH_SIZE = 4
EPOCHS_STAGE_1 = 20
EPOCHS_STAGE_2 = 15

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATASET exists:", os.path.exists(RAW_DATASET))


PROJECT_ROOT: d:\predictions\MRI
RAW_DATASET exists: True


In [47]:
print("Raw dataset contents:")
print(os.listdir(RAW_DATASET))


Raw dataset contents:
['Non PD Patients', 'PD Patients']


In [48]:
def convert_to_png(base_dir):
    converted = 0
    for root, _, files in os.walk(base_dir):
        for f in files:
            src = os.path.join(root, f)

            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                continue

            try:
                img = Image.open(src)
                img.save(src + ".png")
                converted += 1
            except:
                pass  # ignore unreadable files

    print(f"Conversion done. New PNG files: {converted}")

convert_to_png(RAW_DATASET)


Conversion done. New PNG files: 0


In [49]:
if os.path.exists(SPLIT_DATASET):
    shutil.rmtree(SPLIT_DATASET)
    print("Old dataset_split removed")

os.makedirs(SPLIT_DATASET, exist_ok=True)


Old dataset_split removed


In [51]:
SPLITS = {"train": 0.7, "val": 0.15, "test": 0.15}
CLASS_MAP = {
    "NonPD": "Healthy",
    "PD": "Parkinson"
}

random.seed(42)

# Create folders
for split in SPLITS:
    for cls in CLASS_MAP.values():
        os.makedirs(os.path.join(SPLIT_DATASET, split, cls), exist_ok=True)

# Split data
for src_cls, dst_cls in CLASS_MAP.items():
    images = []

    for root, _, files in os.walk(os.path.join(RAW_DATASET, src_cls)):
        for f in files:
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                images.append(os.path.join(root, f))

    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * SPLITS["train"])
    n_val = int(n_total * SPLITS["val"])

    split_map = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train + n_val:]
    }

    for split, files in split_map.items():
        for img in files:
            shutil.copy(
                img,
                os.path.join(
                    SPLIT_DATASET,
                    split,
                    dst_cls,
                    f"{dst_cls}_{os.path.basename(img)}"
                )
            )

print("dataset_split created successfully")


dataset_split created successfully


In [52]:
print("Train Healthy images:",
      len(os.listdir(os.path.join(SPLIT_DATASET, "train", "Healthy"))))
print("Train Parkinson images:",
      len(os.listdir(os.path.join(SPLIT_DATASET, "train", "Parkinson"))))


Train Healthy images: 0
Train Parkinson images: 0


In [53]:
TRAIN_DIR = os.path.join(SPLIT_DATASET, "train")
VAL_DIR = os.path.join(SPLIT_DATASET, "val")

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.


In [54]:
import os
from collections import Counter

RAW_DATASET = os.path.abspath(os.path.join(os.getcwd(), "..", "dataset"))

exts = []

for root, _, files in os.walk(RAW_DATASET):
    for f in files:
        exts.append(os.path.splitext(f)[1].lower())

print("File extensions found:")
print(Counter(exts))


File extensions found:
Counter({'.png': 44007, '.csv': 2, '.txt': 2, '.exe': 2, '.inf': 1, '.xml': 1, '.pdf': 1, '.htm': 1, '.bat': 1})


In [55]:
import nibabel as nib
import numpy as np
from PIL import Image
import os

OUTPUT_PNG_DIR = os.path.join(PROJECT_ROOT, "png_images")
os.makedirs(OUTPUT_PNG_DIR, exist_ok=True)

def normalize(img):
    img = img - np.min(img)
    img = img / (np.max(img) + 1e-8)
    return (img * 255).astype(np.uint8)

for cls in ["PD", "NonPD"]:
    src_dir = os.path.join(RAW_DATASET, cls)
    dst_dir = os.path.join(OUTPUT_PNG_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    for root, _, files in os.walk(src_dir):
        for f in files:
            if f.lower().endswith((".nii", ".nii.gz")):  # MRI volumes
                img = nib.load(os.path.join(root, f)).get_fdata()

                for i in range(img.shape[2]):  # axial slices
                    slice_img = normalize(img[:, :, i])
                    Image.fromarray(slice_img).save(
                        os.path.join(dst_dir, f"{os.path.splitext(f)[0]}_slice_{i}.png")
                    )

print("✅ MRI slices extracted as PNG")


✅ MRI slices extracted as PNG


In [56]:
RAW_DATASET = os.path.join(PROJECT_ROOT, "png_images")
SPLIT_DATASET = os.path.join(PROJECT_ROOT, "dataset_split")

import random, shutil

CLASS_MAP = {"NonPD": "Healthy", "PD": "Parkinson"}
SPLITS = {"train": 0.7, "val": 0.15, "test": 0.15}

if os.path.exists(SPLIT_DATASET):
    shutil.rmtree(SPLIT_DATASET)

random.seed(42)

for split in SPLITS:
    for cls in CLASS_MAP.values():
        os.makedirs(os.path.join(SPLIT_DATASET, split, cls), exist_ok=True)

for src_cls, dst_cls in CLASS_MAP.items():
    images = [
        os.path.join(RAW_DATASET, src_cls, f)
        for f in os.listdir(os.path.join(RAW_DATASET, src_cls))
        if f.endswith(".png")
    ]

    random.shuffle(images)

    n = len(images)
    t = int(0.7 * n)
    v = int(0.85 * n)

    split_map = {
        "train": images[:t],
        "val": images[t:v],
        "test": images[v:]
    }

    for split, imgs in split_map.items():
        for img in imgs:
            shutil.copy(img, os.path.join(SPLIT_DATASET, split, dst_cls))

print("✅ dataset_split created from PNG images")


✅ dataset_split created from PNG images


In [57]:
train_gen = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DATASET, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


Found 0 images belonging to 2 classes.


In [58]:
import os

print("SPLIT_DATASET:", SPLIT_DATASET)
print("Exists:", os.path.exists(SPLIT_DATASET))

for cls in ["Healthy", "Parkinson"]:
    path = os.path.join(SPLIT_DATASET, "train", cls)
    print(f"\n{cls} train folder exists:", os.path.exists(path))
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"{cls} file count:", len(files))
        print(f"{cls} first 10 entries:", files[:10])


SPLIT_DATASET: d:\predictions\MRI\dataset_split
Exists: True

Healthy train folder exists: True
Healthy file count: 0
Healthy first 10 entries: []

Parkinson train folder exists: True
Parkinson file count: 0
Parkinson first 10 entries: []


In [59]:
import os
from collections import Counter

RAW_DATASET = r"d:\predictions\MRI\dataset"

exts = []
for root, _, files in os.walk(RAW_DATASET):
    for f in files:
        exts.append(os.path.splitext(f)[1].lower())

print(Counter(exts))


Counter({'.png': 44007, '.csv': 2, '.txt': 2, '.exe': 2, '.inf': 1, '.xml': 1, '.pdf': 1, '.htm': 1, '.bat': 1})


In [2]:
import os
import shutil

SRC_ROOT = r"D:\predictions\MRI\dataset"
DST_ROOT = r"D:\predictions\MRI\png_dataset"

CLASS_MAP = {
    "Non PD Patients": "NonPD",
    "PD Patients": "PD"
}

for src_class, dst_class in CLASS_MAP.items():
    src_path = os.path.join(SRC_ROOT, src_class)
    dst_path = os.path.join(DST_ROOT, dst_class)
    os.makedirs(dst_path, exist_ok=True)

    for root, _, files in os.walk(src_path):
        for f in files:
            if f.lower().endswith(".png"):
                src_file = os.path.join(root, f)

                # create unique name using folder path
                rel = os.path.relpath(root, src_path)
                rel = rel.replace("\\", "_").replace(" ", "_")
                new_name = f"{rel}_{f}"

                shutil.copy(src_file, os.path.join(dst_path, new_name))

print("✅ PNG files flattened successfully")


✅ PNG files flattened successfully


In [4]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


In [5]:
IMG_SIZE = (192, 192)
BATCH_SIZE = 4

TRAIN_DIR = r"D:\predictions\MRI\dataset_split\train"
VAL_DIR   = r"D:\predictions\MRI\dataset_split\val"

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists:", os.path.exists(VAL_DIR))


Train exists: True
Val exists: True


In [6]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)


In [7]:
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


Found 0 images belonging to 2 classes.


In [8]:
val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


Found 0 images belonging to 2 classes.


In [9]:
import os

PNG_DATASET = r"D:\predictions\MRI\png_dataset"

print("NonPD PNGs:", len(os.listdir(os.path.join(PNG_DATASET, "NonPD"))))
print("PD PNGs:", len(os.listdir(os.path.join(PNG_DATASET, "PD"))))


NonPD PNGs: 10711
PD PNGs: 33296


In [10]:
import shutil

SPLIT_DATASET = r"D:\predictions\MRI\dataset_split"

if os.path.exists(SPLIT_DATASET):
    shutil.rmtree(SPLIT_DATASET)
    print("Old dataset_split deleted")


Old dataset_split deleted


In [11]:
import random
import os
import shutil

PNG_DATASET = r"D:\predictions\MRI\png_dataset"
SPLIT_DATASET = r"D:\predictions\MRI\dataset_split"

CLASS_MAP = {
    "NonPD": "Healthy",
    "PD": "Parkinson"
}

random.seed(42)

# Create folders
for split in ["train", "val", "test"]:
    for cls in CLASS_MAP.values():
        os.makedirs(os.path.join(SPLIT_DATASET, split, cls), exist_ok=True)

# Split images
for src_cls, dst_cls in CLASS_MAP.items():
    images = [
        os.path.join(PNG_DATASET, src_cls, f)
        for f in os.listdir(os.path.join(PNG_DATASET, src_cls))
        if f.lower().endswith(".png")
    ]

    random.shuffle(images)

    n = len(images)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split, imgs in splits.items():
        for img in imgs:
            shutil.copy(
                img,
                os.path.join(SPLIT_DATASET, split, dst_cls)
            )

print("✅ dataset_split recreated from png_dataset")


✅ dataset_split recreated from png_dataset


In [12]:
for cls in ["Healthy", "Parkinson"]:
    path = os.path.join(SPLIT_DATASET, "train", cls)
    print(cls, "train count:", len(os.listdir(path)))


Healthy train count: 7497
Parkinson train count: 23307


In [13]:
train_gen = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DATASET, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


Found 30804 images belonging to 2 classes.


In [14]:
val_gen = val_datagen.flow_from_directory(
    os.path.join(SPLIT_DATASET, "val"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


Found 6601 images belonging to 2 classes.


In [15]:
print(train_gen.class_indices)


{'Healthy': 0, 'Parkinson': 1}


In [16]:
import numpy as np

labels = train_gen.classes
num_healthy = np.sum(labels == 0)
num_parkinson = np.sum(labels == 1)
total = len(labels)

class_weights = {
    0: total / (2 * num_healthy),
    1: total / (2 * num_parkinson)
}

print("Healthy:", num_healthy)
print("Parkinson:", num_parkinson)
print("Class weights:", class_weights)


Healthy: 7497
Parkinson: 23307
Class weights: {0: 2.054421768707483, 1: 0.6608315098468271}


In [17]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model


In [18]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # Stage 1: freeze backbone

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 192, 192, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 96, 96, 32)           864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 96, 96, 32)           128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 96, 96, 32)           0         ['bn_Conv1[0][0]']        

In [20]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [21]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]


In [25]:
history_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)


Epoch 1/15
7701/7701 [==============================] - 2167s 281ms/step - loss: 0.2563 - accuracy: 0.8833 - val_loss: 0.2856 - val_accuracy: 0.8702 - lr: 9.0000e-06
Epoch 2/15
7701/7701 [==============================] - 1463s 190ms/step - loss: 0.2539 - accuracy: 0.8833 - val_loss: 0.2836 - val_accuracy: 0.8685 - lr: 9.0000e-06
Epoch 3/15
7701/7701 [==============================] - 1736s 225ms/step - loss: 0.2505 - accuracy: 0.8856 - val_loss: 0.2776 - val_accuracy: 0.8723 - lr: 9.0000e-06
Epoch 4/15
7701/7701 [==============================] - 1960s 254ms/step - loss: 0.2532 - accuracy: 0.8847 - val_loss: 0.2832 - val_accuracy: 0.8690 - lr: 9.0000e-06
Epoch 5/15
7701/7701 [==============================] - 1543s 200ms/step - loss: 0.2486 - accuracy: 0.8865 - val_loss: 0.2803 - val_accuracy: 0.8696 - lr: 9.0000e-06
Epoch 6/15
7701/7701 [==============================] - 1872s 243ms/step - loss: 0.2486 - accuracy: 0.8850 - val_loss: 0.2750 - val_accuracy: 0.8723 - lr: 9.0000e-06
Epoc

In [26]:
model.save("mri_parkinson_final.keras")
print("✅ Model saved as mri_parkinson_final.keras")


✅ Model saved as mri_parkinson_final.keras
